# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [53]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [54]:
# Import the necessary libs
import os
from dotenv import load_dotenv, find_dotenv
import json
from typing import List, Dict, Any, Optional, Literal
from pydantic import BaseModel, Field
import chromadb
from chromadb.utils import embedding_functions
from tavily import TavilyClient

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

In [55]:
load_dotenv(find_dotenv(), override=True)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"
assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY is not set"

In [56]:
# Environment variables are loaded in the setup cell above.

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError(
        'OPENAI_API_KEY not found in .env file. '
        'Create a .env file with OPENAI_API_KEY="your_key".'
    )

if not TAVILY_API_KEY:
    raise ValueError(
        'TAVILY_API_KEY not found in .env file. '
        'Create a .env file with TAVILY_API_KEY="your_key".'
    )

In [57]:
# VectorDB setup
chroma_client = chromadb.PersistentClient(path="chromadb")

embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key_env_var="OPENAI_API_KEY",
    api_base=os.getenv("OPENAI_BASE_URL"),
)

collection = chroma_client.get_or_create_collection(
    name="udaplay",
    embedding_function=embedding_fn
)

print(f"Collection '{collection.name}' is ready.")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Collection 'udaplay' is ready.


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [58]:
# Create retrieve_game tool
@tool
def retrieve_game(query: str) -> List[Dict[str, Any]]:
    """Semantic search: Finds most results in the vector DB.

    args:
    - query: a question about game industry.

    You'll receive results as list. Each element contains:
    - Platform: like Game Boy, Playstation 5, Xbox 360...)
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game
    """

    results = collection.query(query_texts=[query], n_results=5)
    formatted_results = []

    if results and results['metadatas'] and len(results['metadatas']) > 0:
        for idx_list in results['metadatas']:
            for meta in idx_list:
                formatted_results.append({
                    "Platform": meta.get("Platform", "Unknown"),
                    "Name": meta.get("Name", "Unknown"),
                    "YearOfRelease": meta.get("YearOfRelease", "Unknown"),
                    "Description": meta.get("Description", "No available description")
                })
    return formatted_results


#### Evaluate Retrieval Tool

In [59]:
# Evaluation schema
class EvaluationReport(BaseModel):
    """Data model for evaluation report."""
    useful: bool = Field(description="whether the documents are useful to answer the question")
    description: str = Field(description="description about the evaluation result")

# Create evaluate_retrieval tool
@tool
def evaluate_retrieval(question: str, retrieved_docs: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.

    args:
        question: Original question from the user.
        retrieved_docs: Documents most similar to the user query in the Vector Database.

    The result includes:
            - useful (bool): whether the docs are sufficient to answer the question
            - description (str): description about the evaluation result
    """
    llm = LLM(model="gpt-4o-mini")

    llm_prompt = (
        "Evaluate whether a set of documents are sufficient "
        "to answer the user's query.\n\n"
        f"User query: {question}\n\n"
        f"Documents (JSON):\n{json.dumps(retrieved_docs, indent=2)}\n\n"
        "Provide an evaluation report that includes:\n"
        " - useful (bool): True or False - whether the docs are sufficient to answer the question\n"
        " - description (str): detailed description about your evaluation.\n"
    )

    llm_response = llm.invoke(llm_prompt, response_format=EvaluationReport)

    # Parse llm response
    try:
        parsed = EvaluationReport.model_validate_json(llm_response.content)
        return parsed.model_dump()
    except Exception:
        # Fallback: keep the conversation moving even if parsing fails.
        return {
            "useful": False,
            "description": "LLM response could not be parsed.",
        }

#### Game Web Search Tool

In [60]:
# Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Web search: searches the web with Tavily when the vector DB results aren't good enough
#    args:
#    - question: a question about game industry. 
#
#    Returns the search results, including each page's URL, so the answer can cite its source.
# Tavily client
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

@tool
def game_web_search(query: str) -> List[Dict[str, str]]:
    """
    Semantic search: Finds most results in the vector DB

    args:
        question: a question about the game industry.

    Returns:
        List[Dict[str, str]]: List of relevant search results from the web.
            Each result contains:
                - title: Title of the result
                - url: URL of the result
                - content: A short snippet or summary of the result
    """

    try:
        response = tavily_client.search(
            query,
            search_depth="basic",
            max_results=5,
        )
    except Exception as e:
        return f"Error with Tavily search: {str(e)}"

    results = []
    for result in response.get("results", []):
        results.append({
            "title": result.get("title", "Untitled"),
            "url": result.get("url", ""),
            "content": result.get("content", "No available content.")
        })

    return results

### Agent

In [61]:
# Create your Agent abstraction using StateMachine
udaplay_instructions = """
You are UdaPlay, an agentic AI gaming industry expert. 
Answer the user's question clearly and concisely by following the steps in this exact execution pipeline: 
1. Pull context using the retrieve_game tool.
2. Assess the data with evaluate_retrieval.
3. If the retrieved data is insufficient, refine your query or fall back to the game_web_search tool.
4. Synthesize all useful data into a clean, structured response.
Always explicitly explain your tool-selection reasoning, format the final answer using clean markdown, cite all sources, and maintain an authoritative, analytical tone.
"""

agent = Agent(
    model_name="gpt-4o-mini",
    instructions=udaplay_instructions,
    tools=[retrieve_game, evaluate_retrieval, game_web_search]
)
print("UdaPlay agent ready.")

UdaPlay agent ready.


In [62]:
# Format agent run
def agent_run(query: str):
    """Pretty-print finished agent run and return structured trace."""
    print("=" * 50)
    print(f"----- Running agent for: '{query}' -----\n")

    # Reset agent's session to start fresh
    agent.reset_session()
    
    # Invoke method returns Run object
    run_object = agent.invoke(query)
    final_state = run_object.get_final_state()
    
    # Show agent's reasoning and tool usage
    messages = final_state["messages"]

    for i, msg in enumerate(messages):
        if msg.role == "assistant":
            # Show tool calls
            if hasattr(msg, 'tool_calls') and msg.tool_calls:
                # Generate specific planning message based on tool calls
                print(f"Agent working:")
                if msg.content:
                    print(f"{msg.content}\n")
                
                # Show each tool call
                for tool_call in msg.tool_calls:
                    print(f"Tool Call: {tool_call.function.name}")
                    print(f"Arguments: {tool_call.function.arguments}\n")
            
            # Show assistant's reasoning when no tools are called (final answer)
            elif msg.content:
                print(f"Agent working:\n{msg.content}\n")
        
        elif msg.role == "tool":
            # Show tool results
            print(f"Tool Result ({msg.name}):")
            print(f"{msg.content[:200]}{'...' if len(msg.content) > 200 else ''}\n")
    
    # Final answer is last in list
    final_answer = messages[-1]

    if final_answer.role == "assistant":
        print("-" * 50)
        print(f"FINAL ANSWER:")
        print(final_answer.content)
        print("=" * 50)
    else:
        print("Agent could not generate an answer.")
    print(f"\n----- Run complete. Total tokens used: {final_state.get('total_tokens', 0)} -----\n")

In [66]:
# Invoke your agent
# - When was Pokémon Gold and Silver released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X released for PlayStation 5?
agent_run("When was Pokémon Gold and Silver released?")

----- Running agent for: 'When was Pokémon Gold and Silver released?' -----

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Agent working:
Tool Call: retrieve_game
Arguments: {"query":"Pokémon Gold and Silver release date"}

Tool Result (retrieve_game):
"[{'Platform': 'Game Boy Color', 'Name': 'Pok\u00e9mon Gold and Silver', 'YearOfRelease': 1999, 'Description': 'Second-generation Pok\u00e9mon games introducing new regions, Pok\u00e9mon, and gameplay...

Agent working:
Tool Call: evaluate_retrieval
Arguments: {"question":"When was Pokémon Gold and Silver released?","retrieved_docs":[{"Platform":"Game Boy Color","Name":"Pokémon Gold and Silver","YearOfRelease":1999,"Description":"Second-gene

In [67]:
agent_run("Which one was the first 3D platformer Mario game?")

----- Running agent for: 'Which one was the first 3D platformer Mario game?' -----

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Agent working:
Tool Call: retrieve_game
Arguments: {"query":"first 3D platformer Mario game"}

Tool Result (retrieve_game):
"[{'Platform': 'Nintendo 64', 'Name': 'Super Mario 64', 'YearOfRelease': 1996, 'Description': \"A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue P...

Agent working:
Tool Call: evaluate_retrieval
Arguments: {"question":"Which one was the first 3D platformer Mario game?","retrieved_docs":[{"Platform":"Nintendo 64","Name":"Super Mario 64","YearOfRelease":1996,"Description":"A groundbreakin

In [69]:
agent_run("Was Mortal Kombat X released for PlayStation 5?")

----- Running agent for: 'Was Mortal Kombat X released for PlayStation 5?' -----

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Agent working:
Tool Call: retrieve_game
Arguments: {"query":"Mortal Kombat X PlayStation 5 release"}

Tool Result (retrieve_game):
"[{'Platform': 'PlayStation 5', 'Name': \"Marvel's Spider-Man 2\", 'YearOfRelease': 2023, 'Description': 'The sequel to the acclaimed Spider-Man game, featuring both Peter Parker and Miles Morales as ...

Agent working:
Tool Call: evaluate_retrieval
Arguments: {"question":"Was Mortal Kombat X released for PlayStation 5?","retrieved_docs":[{"Platfo